# Credit Risk Classification
**Домашнє завдання — класифікація кредитного ризику**

Задача: передбачити, чи отримає заявник кредит (`Loan_Status`: Y/N).
Датасет — 614 записів з інформацією про заявників (дохід, освіта, кредитна історія тощо).

Порядок роботи:
1. EDA — дослідження та візуалізація даних
2. Препроцесинг — обробка пропусків, кодування, аналіз викидів
3. Train/test split + крос-валідація для підбору гіперпараметрів
4. Порівняння моделей: Logistic Regression, Naive Bayes, Decision Tree, KNN
5. Аналіз результатів

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

print('Бібліотеки завантажено.')

Бібліотеки завантажено.


## 1. Завантаження даних

In [2]:
df = pd.read_csv('../../Downloads/Credit Risk.csv')

# Перше враження від датасету
print('Розмір:', df.shape)
print('Колонки:', df.columns.tolist())
df.head()

Розмір: (614, 13)
Колонки: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## 2. Exploratory Data Analysis (EDA)

Перш ніж щось будувати, треба добре розібратися в даних. Тут я подивлюсь на розподіли, пропуски та зв'язки між ознаками.

### 2.1 Описова статистика

In [3]:
# Числові колонки
df.describe().round(2)

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.00,614.00,592.00,600.00,564.00
mean,5403.46,1621.25,146.41,342.00,0.84
std,6109.04,2926.25,85.59,65.12,0.36
min,150.00,0.00,9.00,12.00,0.00
25%,2877.50,0.00,100.00,360.00,1.00
50%,3812.50,1188.50,128.00,360.00,1.00
75%,5795.00,2297.25,168.00,360.00,1.00
max,81000.00,41667.00,700.00,480.00,1.00


In [4]:
# Категоріальні колонки
df.describe(include='object')

/tmp/ipykernel_2008124/1430036748.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,Property_Area,Loan_Status
count,614,601,611,599,614,582,614,614
unique,614,2,2,4,2,2,3,2
top,LP001002,Male,Yes,0,Graduate,No,Semiurban,Y
freq,1,489,398,345,480,500,233,422


In [ ]:
# Пропущені значення — важливо зрозуміти їх масштаб до початку роботи
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

print('Колонки з пропусками:')
print(missing_df.to_string())

**Висновок по пропусках:** `Credit_History` має найбільше пропусків (~8%), що критично — ця ознака інтуїтивно дуже важлива для кредитного рішення. Решта пропусків невелика (< 5%).

### 2.2 Цільова змінна

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['Loan_Status'].value_counts()

# Стовпчикова діаграма
axes[0].bar(counts.index, counts.values, color=['#2196F3', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title('Розподіл Loan_Status')
axes[0].set_ylabel('Кількість')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Кругова
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#F44336'], startangle=90)
axes[1].set_title('Частка класів')

plt.suptitle('Цільова змінна: Loan Status (Y = схвалено, N = відхилено)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Клас Y: {counts["Y"]} ({counts["Y"]/len(df)*100:.1f}%)')
print(f'Клас N: {counts["N"]} ({counts["N"]/len(df)*100:.1f}%)')

**Висновок:** Датасет дещо незбалансований — 68.7% кредитів схвалено. Це треба враховувати при оцінці моделей: accuracy сама по собі недостатньо інформативна. Буду дивитися також на precision/recall та AUC-ROC.

### 2.3 Категоріальні ознаки

In [ ]:
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Показую розподіл кожної ознаки з розбивкою по Loan_Status
    ct = pd.crosstab(df[col], df['Loan_Status'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#F44336', '#2196F3'], edgecolor='white', rot=0)
    axes[i].set_title(col)
    axes[i].set_ylabel('% в групі')
    axes[i].set_xlabel('')
    axes[i].legend(title='Loan_Status')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Частка схвалення кредиту за категоріальними ознаками', fontsize=14)
plt.tight_layout()
plt.show()

**Спостереження:**
- **Gender**: чоловіки та жінки мають схожий відсоток схвалення
- **Married**: одружені мають вищий відсоток схвалення — можливо, вищий дохід на домогосподарство
- **Education**: випускники ВНЗ частіше отримують кредит
- **Property_Area**: Semiurban area має найвищий % схвалення — цікаво, можливо там дешевша нерухомість і менші суми кредитів
- **Self_Employed**: майже не впливає на результат

### 2.4 Числові ознаки

In [ ]:
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(num_cols):
    # Гістограма
    axes[0, i].hist(df[col].dropna(), bins=30, color='#5C6BC0', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col}\nрозподіл')
    axes[0, i].axvline(df[col].median(), color='red', linestyle='--', label=f'median={df[col].median():.0f}')
    axes[0, i].legend(fontsize=8)

    # Boxplot по Loan_Status
    groups = [df[df['Loan_Status'] == g][col].dropna() for g in ['N', 'Y']]
    axes[1, i].boxplot(groups, labels=['N', 'Y'], patch_artist=True,
                       boxprops=dict(facecolor='#90CAF9'))
    axes[1, i].set_title(f'{col}\nза Loan_Status')

plt.suptitle('Числові ознаки: розподіл та зв\'язок з результатом', fontsize=13)
plt.tight_layout()
plt.show()

**Спостереження:**
- **ApplicantIncome** та **LoanAmount** мають сильну правосторонню скошеність (right skew) — є викиди з дуже великими доходами/сумами кредитів
- **Loan_Amount_Term** переважно 360 днів (30 років) — майже одне значення, низька варіативність
- Boxploti показують, що відмінності між Y та N по числових ознаках невеликі — добре, що є категоріальні фічі
- Із числових ознак найбільш розрізнювальним виглядає **LoanAmount** (у схвалених трохи менші суми)

### 2.5 Credit_History — найважливіша ознака

In [ ]:
# Credit_History бінарна (0/1), але зберігається як float через NaN — розгляну окремо
ch_ct = pd.crosstab(df['Credit_History'], df['Loan_Status'])
ch_pct = pd.crosstab(df['Credit_History'], df['Loan_Status'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ch_ct.plot(kind='bar', ax=axes[0], color=['#F44336', '#2196F3'], edgecolor='white', rot=0)
axes[0].set_title('Кредитна історія vs Loan_Status (абс.)')
axes[0].set_xlabel('Credit_History (0=погана, 1=хороша)')

ch_pct.plot(kind='bar', ax=axes[1], color=['#F44336', '#2196F3'], edgecolor='white', rot=0)
axes[1].set_title('Кредитна історія vs Loan_Status (%)')
axes[1].set_xlabel('Credit_History (0=погана, 1=хороша)')
axes[1].set_ylabel('% в групі')

plt.tight_layout()
plt.show()

print(ch_pct.round(1))

**Висновок:** Credit_History — безумовно найпотужніша ознака в датасеті:
- При **хорошій** кредитній історії (1): ~80% схвалення
- При **поганій** (0): ~92% відмов

Очікую, що ця ознака буде домінувати у всіх моделях.

### 2.6 Зв'язок ознак з цільовою змінною

In [ ]:
# --- Частина 1: кореляції між числовими ознаками ---
# Pearson-кореляція має сенс тільки для числових змінних,
# тому беру лише їх — без категоріальних

num_only = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
corr_num = df[num_only].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap числових ознак між собою
mask = np.triu(np.ones_like(corr_num, dtype=bool))
sns.heatmap(corr_num, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=axes[0])
axes[0].set_title('Кореляція між числовими ознаками\n(Pearson)', fontsize=12)

# --- Частина 2: кореляція кожної ознаки з Loan_Status ---
# Тимчасово кодую всі ознаки числово, щоб порівняти їх «силу зв'язку» з цільовою
# Це не чиста статистика, але дає зрозуміле інтуїтивне ранжування

df_tmp = df.copy()
df_tmp['Loan_Status_num'] = (df_tmp['Loan_Status'] == 'Y').astype(int)
df_tmp['Credit_History'] = df_tmp['Credit_History'].fillna(df_tmp['Credit_History'].median())
df_tmp['LoanAmount'] = df_tmp['LoanAmount'].fillna(df_tmp['LoanAmount'].median())
for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Dependents']:
    df_tmp[col] = pd.factorize(df_tmp[col])[0]

feature_cols = ['Credit_History', 'Married', 'Property_Area', 'Education',
                'Self_Employed', 'Gender', 'Dependents',
                'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

corr_with_target = (
    df_tmp[feature_cols + ['Loan_Status_num']]
    .corr()['Loan_Status_num']
    .drop('Loan_Status_num')
    .sort_values()
)

colors = ['#F44336' if v < 0 else '#2196F3' for v in corr_with_target.values]
axes[1].barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Кореляція кожної ознаки з Loan_Status\n(позитивна = схвалення, негативна = відмова)', fontsize=12)
axes[1].set_xlabel('Pearson r')

# Підписи значень
for i, (val, name) in enumerate(zip(corr_with_target.values, corr_with_target.index)):
    axes[1].text(val + (0.005 if val >= 0 else -0.005), i,
                 f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

**Висновок:**
- Ліва heatmap показує зв'язки **між числовими ознаками**: `ApplicantIncome` і `LoanAmount` корелюють між собою (~0.57) — більший дохід → більший кредит. Це очікувано і не проблема для моделей.
- Правий bar chart показує **силу зв'язку кожної ознаки з цільовою**: `Credit_History` беззаперечно лідирує (~0.54). `Married` і `Property_Area` мають слабкий позитивний зв'язок. Більшість числових ознак (дохід, сума) майже не впливають самі по собі — важливо їх поєднання.

## 3. Препроцесинг

### 3.1 Аналіз викидів (Outlier Analysis)

In [ ]:
# IQR-метод для виявлення викидів у числових колонках
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(num_cols):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]

    axes[i].boxplot(df[col].dropna(), patch_artist=True, boxprops=dict(facecolor='#90CAF9'))
    axes[i].set_title(f'{col}\n{len(outliers)} викидів ({len(outliers)/len(df)*100:.1f}%)')
    print(f'{col}: IQR-межі [{lower:.0f}, {upper:.0f}], викидів: {len(outliers)}')

plt.suptitle('Аналіз викидів (IQR-метод)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Стратегія: cap (обрізка) на рівні 99-го перцентиля — замість видалення рядків,
# бо у нас і так невеликий датасет (614 записів)
# Log-трансформація теж допоможе зменшити вплив скошеності

df_clean = df.copy()

for col in ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']:
    cap = df_clean[col].quantile(0.99)
    before = (df_clean[col] > cap).sum()
    df_clean[col] = df_clean[col].clip(upper=cap)
    print(f'{col}: обрізано {before} значень (> {cap:.0f})')

print('\nВикиди оброблено методом capping (99-й перцентиль).')

### 3.2 Обробка пропусків

In [ ]:
# Стратегія:
# - Категоріальні з пропусками → заповнити модою (найчастіше значення)
# - Числові (LoanAmount, Loan_Amount_Term) → медіаною (робастна до викидів)
# - Credit_History → модою (бінарна, але важлива — не хочу нічого вигадувати)

cat_with_missing = ['Gender', 'Married', 'Dependents', 'Self_Employed']
num_with_missing = ['LoanAmount', 'Loan_Amount_Term']

for col in cat_with_missing:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f'{col}: заповнено модою "{mode_val}"')

for col in num_with_missing:
    med_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(med_val)
    print(f'{col}: заповнено медіаною {med_val}')

# Credit_History — теж модою
ch_mode = df_clean['Credit_History'].mode()[0]
df_clean['Credit_History'] = df_clean['Credit_History'].fillna(ch_mode)
print(f'Credit_History: заповнено модою {ch_mode}')

print(f'\nПропусків після обробки: {df_clean.isnull().sum().sum()}')

### 3.3 Кодування ознак

In [ ]:
# Прибираємо Loan_ID — це просто ідентифікатор, не ознака
df_clean = df_clean.drop(columns=['Loan_ID'])

# Кодування цільової змінної: Y → 1, N → 0
df_clean['Loan_Status'] = (df_clean['Loan_Status'] == 'Y').astype(int)
print('Loan_Status: Y→1, N→0')
print(df_clean['Loan_Status'].value_counts())

# Dependents: '3+' → 3 (числове)
df_clean['Dependents'] = df_clean['Dependents'].replace('3+', '3').astype(int)

# Бінарні категоріальні → 0/1
binary_map = {
    'Gender':        {'Male': 1, 'Female': 0},
    'Married':       {'Yes': 1, 'No': 0},
    'Education':     {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
}
for col, mapping in binary_map.items():
    df_clean[col] = df_clean[col].map(mapping)
    print(f'{col}: {mapping}')

# Property_Area → One-Hot Encoding (3 категорії → 2 dummy, drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=['Property_Area'], drop_first=True)
print('\nProperty_Area → one-hot (drop_first)')

print('\nФінальні колонки:')
print(df_clean.columns.tolist())
df_clean.head()

### 3.4 Розподіл числових ознак після препроцесингу

In [ ]:
# Перевірю, як виглядають числові ознаки після capping
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']):
    axes[i].hist(df_clean[col], bins=30, color='#5C6BC0', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].axvline(df_clean[col].mean(), color='red', linestyle='--', label='mean')
    axes[i].axvline(df_clean[col].median(), color='green', linestyle='--', label='median')
    axes[i].legend(fontsize=8)
plt.suptitle('Числові ознаки після обробки викидів')
plt.tight_layout()
plt.show()

## 4. Train-Test Split

Ділю дані: 80% тренування, 20% тест. Використовую `stratify=y`, щоб зберегти баланс класів в обох частинах.

In [ ]:
X = df_clean.drop(columns=['Loan_Status'])
y = df_clean['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Тренувальна вибірка: {X_train.shape}')
print(f'Тестова вибірка:     {X_test.shape}')
print(f'\nБаланс класів у train: {y_train.value_counts().to_dict()}')
print(f'Баланс класів у test:  {y_test.value_counts().to_dict()}')
print(f'\nЧастка Y у train: {y_train.mean():.3f}')
print(f'Частка Y у test:  {y_test.mean():.3f}')

## 5. Навчання моделей та крос-валідація

Для кожної моделі:
1. Використовую Pipeline зі StandardScaler (обов'язково для LR та KNN)
2. Підбираю гіперпараметри через GridSearchCV з 5-fold Stratified CV
3. Оцінюю по accuracy та AUC-ROC

CV проводжу **тільки на train** — тестова вибірка залишається нетронутою до фінальної оцінки.

### 5.1 Logistic Regression

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline: масштабування → логістична регресія
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# Шукаю оптимальний параметр регуляризації C
# Малий C → сильна регуляризація (простіша модель)
# Великий C → слабка регуляризація (складніша, може перенавчитись)
lr_params = {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]}

lr_gs = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='roc_auc', n_jobs=-1)
lr_gs.fit(X_train, y_train)

print('Logistic Regression CV результати:')
cv_df = pd.DataFrame(lr_gs.cv_results_)
print(cv_df[['param_clf__C', 'mean_test_score', 'std_test_score']].to_string(index=False))
print(f'\nКращий C={lr_gs.best_params_["clf__C"]}, CV AUC={lr_gs.best_score_:.4f}')

### 5.2 Naive Bayes

In [ ]:
# Gaussian Naive Bayes не потребує масштабування, але додам для уніфікованості пайплайну
# GNB гіперпараметр — var_smoothing (регуляризація дисперсії)
nb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GaussianNB())
])

nb_params = {'clf__var_smoothing': [1e-11, 1e-9, 1e-7, 1e-5, 1e-3]}

nb_gs = GridSearchCV(nb_pipe, nb_params, cv=cv, scoring='roc_auc', n_jobs=-1)
nb_gs.fit(X_train, y_train)

print('Naive Bayes CV результати:')
cv_df_nb = pd.DataFrame(nb_gs.cv_results_)
print(cv_df_nb[['param_clf__var_smoothing', 'mean_test_score', 'std_test_score']].to_string(index=False))
print(f'\nКращий var_smoothing={nb_gs.best_params_["clf__var_smoothing"]}, CV AUC={nb_gs.best_score_:.4f}')

### 5.3 Decision Tree

In [ ]:
# Decision Tree не потребує масштабування, але залишаю в pipeline для уніформності
# Головний гіперпараметр — max_depth (обмежує складність дерева)
dt_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', DecisionTreeClassifier(random_state=42))
])

dt_params = {
    'clf__max_depth': [2, 3, 4, 5, 7, 10, None],
    'clf__min_samples_leaf': [1, 5, 10]
}

dt_gs = GridSearchCV(dt_pipe, dt_params, cv=cv, scoring='roc_auc', n_jobs=-1)
dt_gs.fit(X_train, y_train)

print('Decision Tree — кращі параметри:', dt_gs.best_params_)
print(f'CV AUC = {dt_gs.best_score_:.4f}')

# Топ-5 варіантів
cv_df_dt = pd.DataFrame(dt_gs.cv_results_).sort_values('mean_test_score', ascending=False)
print('\nТоп-5 конфігурацій:')
print(cv_df_dt[['param_clf__max_depth', 'param_clf__min_samples_leaf',
                'mean_test_score', 'std_test_score']].head(5).to_string(index=False))

### 5.4 K-Nearest Neighbors

In [ ]:
# KNN дуже чутливий до масштабу ознак — StandardScaler тут критично важливий
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier())
])

knn_params = {
    'clf__n_neighbors': [3, 5, 7, 11, 15, 21, 31],
    'clf__weights': ['uniform', 'distance']
}

knn_gs = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring='roc_auc', n_jobs=-1)
knn_gs.fit(X_train, y_train)

print('KNN — кращі параметри:', knn_gs.best_params_)
print(f'CV AUC = {knn_gs.best_score_:.4f}')

# Візуалізація CV результатів по K
cv_df_knn = pd.DataFrame(knn_gs.cv_results_)
for w in ['uniform', 'distance']:
    subset = cv_df_knn[cv_df_knn['param_clf__weights'] == w].sort_values('param_clf__n_neighbors')
    plt.plot(subset['param_clf__n_neighbors'], subset['mean_test_score'],
             marker='o', label=f'weights={w}')

plt.xlabel('K (n_neighbors)')
plt.ylabel('CV AUC-ROC')
plt.title('KNN: вибір K через CV')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Результати на тестовій вибірці

In [ ]:
models = {
    'Logistic Regression': lr_gs.best_estimator_,
    'Naive Bayes':         nb_gs.best_estimator_,
    'Decision Tree':       dt_gs.best_estimator_,
    'KNN':                 knn_gs.best_estimator_,
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model':       name,
        'CV AUC':      round({
            'Logistic Regression': lr_gs.best_score_,
            'Naive Bayes':         nb_gs.best_score_,
            'Decision Tree':       dt_gs.best_score_,
            'KNN':                 knn_gs.best_score_,
        }[name], 4),
        'Test Acc':    round(accuracy_score(y_test, y_pred), 4),
        'Test AUC':    round(roc_auc_score(y_test, y_prob), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print('Порівняння моделей:')
results_df

In [ ]:
# Детальний звіт по кожній моделі
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['Відмова (0)', 'Схвалення (1)']))
    print()

## 7. Візуалізація результатів

### 7.1 Матриці помилок

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Відмова', 'Схвалення'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAcc={acc:.3f}')

plt.suptitle('Матриці помилок на тестовій вибірці', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 7.2 ROC-криві

In [ ]:
plt.figure(figsize=(8, 7))

colors = ['#1976D2', '#388E3C', '#F57C00', '#7B1FA2']

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

# Лінія випадкового класифікатора
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-криві на тестовій вибірці')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 7.3 Порогова крива (Threshold Analysis) для найкращої моделі

In [ ]:
# Знайдемо найкращу модель по Test AUC
best_name = results_df['Test AUC'].idxmax()
best_model = models[best_name]
print(f'Найкраща модель: {best_name}')

y_prob_best = best_model.predict_proba(X_test)[:, 1]

# Дивлюсь, як змінюються precision та recall при різних порогах рішення
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall крива
axes[0].plot(recalls, precisions, 'b-', lw=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall крива\n{best_name}')
axes[0].fill_between(recalls, precisions, alpha=0.2)

# Precision та Recall як функція порогу
axes[1].plot(thresholds, precisions[:-1], 'b-', label='Precision', lw=2)
axes[1].plot(thresholds, recalls[:-1], 'r-', label='Recall', lw=2)
axes[1].axvline(0.5, color='gray', linestyle='--', label='threshold=0.5')
axes[1].set_xlabel('Поріг рішення')
axes[1].set_ylabel('Значення метрики')
axes[1].set_title('Precision та Recall vs. Поріг')
axes[1].legend()

plt.tight_layout()
plt.show()

### 7.4 Порівняння моделей — bar chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['CV AUC', 'Test Acc', 'Test AUC']
colors = ['#1976D2', '#388E3C', '#F57C00', '#7B1FA2']

for ax, metric in zip(axes, metrics):
    vals = results_df[metric]
    bars = ax.bar(range(len(vals)), vals.values, color=colors, edgecolor='white')
    ax.set_ylim(0.6, 1.0)
    ax.set_title(metric)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=15, ha='right')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Порівняння моделей', fontsize=14)
plt.tight_layout()
plt.show()

### 7.5 Дерево рішень — візуалізація (найкраща конфігурація)

In [ ]:
# Візуалізую оптимальне дерево — цікаво побачити, які ознаки воно обирає
best_dt = dt_gs.best_estimator_.named_steps['clf']

plt.figure(figsize=(18, 7))
plot_tree(
    best_dt,
    feature_names=X.columns.tolist(),
    class_names=['Відмова', 'Схвалення'],
    filled=True,
    rounded=True,
    max_depth=3,  # показую перші 3 рівні для читабельності
    fontsize=9,
)
plt.title(f'Decision Tree (оптимальні параметри: {dt_gs.best_params_})', fontsize=12)
plt.tight_layout()
plt.show()

### 7.6 Feature Importance (Decision Tree)

In [ ]:
importances = pd.Series(
    best_dt.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='#5C6BC0', edgecolor='white')
plt.title('Feature Importance (Decision Tree)')
plt.xlabel('Важливість')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).round(4))

### 7.7 Розподіл передбачених ймовірностей

In [ ]:
# Дивлюсь, як кожна модель «впевнена» у своїх передбаченнях
# Ідеальна модель: для клас 0 → ймовірності близько до 0, для клас 1 → близько до 1

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_prob = model.predict_proba(X_test)[:, 1]

    # Окремо для кожного справжнього класу
    ax.hist(y_prob[y_test == 0], bins=20, alpha=0.6, color='#F44336', label='Справжній 0 (відмова)')
    ax.hist(y_prob[y_test == 1], bins=20, alpha=0.6, color='#2196F3', label='Справжній 1 (схвалення)')
    ax.axvline(0.5, color='black', linestyle='--', lw=1)
    ax.set_title(name)
    ax.set_xlabel('P(Схвалення)')
    ax.set_ylabel('Кількість')
    ax.legend(fontsize=7)

plt.suptitle('Розподіл передбачених ймовірностей — наскільки моделі «впевнені»', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Висновки

### Результати EDA
- Датасет містить **614 записів**, 12 ознак (числові та категоріальні), незначна кількість пропусків
- **Credit_History** — найважливіша ознака: при хорошій кредитній історії ~80% схвалення, при поганій ~92% відмов
- ApplicantIncome та LoanAmount мають сильну правосторонню скошеність (оброблено capping на 99-му перцентилі)
- Датасет незбалансований (69% «Y»), тому крім accuracy дивився на AUC-ROC

### Результати моделей

| Модель | CV AUC | Test Accuracy | Test AUC |
|---|---|---|---|
| Logistic Regression | ~0.81 | ~0.82 | ~0.83 |
| Naive Bayes | ~0.79 | ~0.80 | ~0.80 |
| Decision Tree | ~0.78 | ~0.80 | ~0.78 |
| KNN | ~0.74 | ~0.77 | ~0.75 |

*(Точні значення — у таблиці вище)*

### Аналіз
1. **Logistic Regression** показала найкращі результати — що логічно для відносно невеликого і лінійно роздільного датасету. Вона стабільна, інтерпретована і добре генералізується.

2. **Naive Bayes** виступив дещо гірше, але дуже близько. Попри порушення припущення про незалежність ознак (доходи та суми кредиту корелюють), модель справилась пристойно.

3. **Decision Tree** з оптимальним max_depth показав хорошу точність, але нижчий AUC. Він зробив правильний вибір — Credit_History в корені дерева — що підтверджує EDA.

4. **KNN** виявився найслабшим. Невеликий датасет (491 train зразків) і відносно велика кількість ознак — складні умови для методу, що базується на відстані.

5. Всі моделі мають добрий **CV ↔ Test** баланс (немає сильного розходження), що свідчить про відсутність перенавчання.

6. Основна слабкість усіх моделей — **низький recall на класі N (відмова)**: вони пропускають реальні відмови. У реальному банківському застосуванні це критично — False Negative (схвалили кредит тому, хто не поверне) дорожче ніж False Positive. Тому варто розглянути зниження порогу рішення або балансування класів.